In [1]:
import nltk
import numpy as np
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.utils import to_categorical

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\kgliv\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [2]:
documents = []
labels = []

for fileid in movie_reviews.fileids():
    documents.append(movie_reviews.raw(fileid))
    labels.append(1 if movie_reviews.categories(fileid)[0] == 'pos' else 0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42)

# Tokenization
vocab_size = 10000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [4]:
from tensorflow.keras.layers import SimpleRNN

# Convert to one-hot (bag-of-words)
X_train_onehot = tokenizer.texts_to_matrix(X_train, mode='binary')
X_test_onehot = tokenizer.texts_to_matrix(X_test, mode='binary')

model_rnn_onehot = Sequential([
    SimpleRNN(64, input_shape=(vocab_size, 1)),
    Dense(1, activation='sigmoid')
])

C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [5]:
model_rnn_onehot.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_rnn_onehot.fit(
    X_train_onehot, y_train,
    epochs=1,
    batch_size=32,
    validation_split=0.2
)

print("RNN One-Hot Accuracy:",
      model_rnn_onehot.evaluate(X_test_onehot, y_test)[1])

40/40 ━━━━━━━━━━━━━━━━━━━━ 197s 5s/step - accuracy: 0.4727 - loss: 0.7168 - val_accuracy: 0.5156 - val_loss: 0.7038
13/13 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.4950 - loss: 0.7045
RNN One-Hot Accuracy: 0.4950000047683716


In [6]:
model_rnn_embed = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

model_rnn_embed.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_rnn_embed.fit(
    X_train_pad, y_train,
    epochs=1,
    batch_size=32,
    validation_split=0.2
)

print("RNN Embedding Accuracy:",
      model_rnn_embed.evaluate(X_test_pad, y_test)[1])

C:\Users\kgliv\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


40/40 ━━━━━━━━━━━━━━━━━━━━ 20s 223ms/step - accuracy: 0.5125 - loss: 0.6972 - val_accuracy: 0.4656 - val_loss: 0.7048
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.4875 - loss: 0.6988
RNN Embedding Accuracy: 0.48750001192092896
